In [ ]:
# Imports
import sys
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

sys.path.append(str(Path().resolve().parent))
from src.s3_utils import upload_df_to_s3, read_df_from_s3, list_s3_files
from src.db_utils import get_engine, create_tables, load_df_to_table
from sqlalchemy import text

load_dotenv(dotenv_path=Path().resolve().parent / ".env")

BUCKET = os.getenv("S3_BUCKET_NAME")
print(f"Bucket : {BUCKET}")
print("Imports OK")

In [ ]:
# Chargement des données locales 
DATA_DIR = Path().resolve().parent / "data" / "raw"

df_cities = pd.read_csv(DATA_DIR / "cities.csv")
df_hotels = pd.read_csv(DATA_DIR / "hotels.csv")

print(f"cities : {df_cities.shape}")
print(f"hotels : {df_hotels.shape}")

In [ ]:
#  — EXTRACT : aperçu et contrôle qualité avant transformation
print("=== cities ===")
display(df_cities.head(3))
print(f"Valeurs manquantes :\n{df_cities.isnull().sum()}\n")

print("=== hotels ===")
display(df_hotels.head(3))
print(f"Valeurs manquantes :\n{df_hotels.isnull().sum()}")

In [ ]:
# Transform cities
df_cities_clean = df_cities.copy()
df_cities_clean = df_cities_clean.drop(columns=["size_plot"], errors="ignore")
df_cities_clean["lat"]           = df_cities_clean["lat"].round(6)
df_cities_clean["lon"]           = df_cities_clean["lon"].round(6)
df_cities_clean["temp_moy"]      = df_cities_clean["temp_moy"].round(2)
df_cities_clean["weather_score"] = df_cities_clean["weather_score"].round(2)

print(f"cities_clean : {df_cities_clean.shape}")
print(f"Colonnes : {list(df_cities_clean.columns)}")
df_cities_clean.head(3)

In [ ]:
df_hotels.isnull().sum()

In [ ]:
# Transform hotels
df_hotels_clean = df_hotels.copy()

# Supprimer lignes sans nom
df_hotels_clean = df_hotels_clean.dropna(subset=["hotel_name"]).reset_index(drop=True)

# Imputer score manquant par médiane de la ville
df_hotels_clean["score"] = df_hotels_clean.groupby("city")["score"].transform(
    lambda x: x.fillna(x.median())
)

# Nettoyer hotel_name
df_hotels_clean["hotel_name"] = df_hotels_clean["hotel_name"].str.strip()

print(f"hotels_clean : {df_hotels_clean.shape}")
print(f"Manquantes :\n{df_hotels_clean.isnull().sum()}")
df_hotels_clean.head(3)

In [ ]:
# Load vers S3
print("Upload vers S3...")
upload_df_to_s3(df_cities_clean, BUCKET, "raw/cities.csv")
upload_df_to_s3(df_hotels_clean, BUCKET, "raw/hotels.csv")

print("\nFichiers dans le bucket :")
for f in list_s3_files(BUCKET, prefix="raw/"):
    print(f"  s3://{BUCKET}/{f}")

In [ ]:
# Vérification lecture S3
df_cities_s3 = read_df_from_s3(BUCKET, "raw/cities.csv")
df_hotels_s3 = read_df_from_s3(BUCKET, "raw/hotels.csv")

assert df_cities_s3.shape == df_cities_clean.shape, "Mismatch cities !"
assert df_hotels_s3.shape == df_hotels_clean.shape, "Mismatch hotels !"
print("OK — données identiques entre local et S3")

In [ ]:
#  Recréer tables RDS + chargement
engine = get_engine()

with engine.connect() as conn:
    print("Connexion OK")

# Recréer les tables avec le bon schéma
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hotels CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS cities CASCADE"))
    conn.commit()

create_tables(engine)

# Charger cities en premier (FK oblige)
load_df_to_table(df_cities_s3, "cities", engine)
load_df_to_table(df_hotels_s3, "hotels", engine)

In [ ]:
# Vérification SQL
engine = get_engine() # si execution partielle notebook
with engine.connect() as conn:
    print("=== Top 5 destinations ===")
    result = conn.execute(text("""
        SELECT rank, city, weather_score, temp_moy, rain_total
        FROM cities ORDER BY rank LIMIT 5
    """))
    display(pd.DataFrame(result.fetchall(), columns=result.keys()))

    print("\n=== Top 20 hôtels dans les Top 5 villes ===")
    result = conn.execute(text("""
        SELECT h.hotel_name, h.score, h.price_eur, h.lat, h.lon, c.city, h.distance
        FROM hotels h
        JOIN cities c ON h.city_id = c.city_id
        WHERE h.city_id IN (
            SELECT city_id FROM cities ORDER BY rank LIMIT 5
        )
        AND h.score IS NOT NULL
        ORDER BY h.score DESC
        LIMIT 20
    """))
    hotels_debug = pd.DataFrame(result.fetchall(), columns=result.keys()) # Debug line
    display(hotels_debug)
     

In [ ]:
# Carte n° 1 : Top 5 destination selon le temps

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sqlalchemy import text
from pathlib import Path


 
with engine.connect() as conn:
    df_top5 = pd.read_sql(text("""
        SELECT city_id, city, lat, lon, weather_score, temp_moy, rank
        FROM cities ORDER BY rank LIMIT 5
    """), conn)


df_top5["size_plot"] = df_top5["temp_moy"] - df_top5["temp_moy"].min() + 2
df_top5["label"]     = df_top5.apply(
    lambda r: f"#{int(r['rank'])} {r['city']}<br>{r['temp_moy']:.1f}°C", axis=1
)

fig1 = px.scatter_mapbox(
    df_top5,
    lat="lat", lon="lon",
    hover_name="city",
    hover_data={
        "temp_moy":      ":.1f",
        "weather_score": ":.2f",
        "rank":          True,
        "lat":           False,
        "lon":           False,
        "size_plot":     False,
        "label":         False
    },
    color="temp_moy",
    size="size_plot",
    size_max=40,
    color_continuous_scale="RdYlBu_r",
    zoom=4,
    center={"lat": df_top5["lat"].mean(), "lon": df_top5["lon"].mean()},
    mapbox_style="carto-positron",
    title="🌤️ Top 5 destinations — température moyenne sur 7 jours"
)

 
fig1.add_trace(go.Scattermapbox(
    lat=df_top5["lat"],
    lon=df_top5["lon"],
    mode="text",
    text=df_top5["label"],
    textposition="top right",
    textfont=dict(size=11, color="black"),
    hoverinfo="skip",
    showlegend=False
))

fig1.update_layout(
    coloraxis_colorbar=dict(title="Temp. moy (°C)"),
    height=900, 
    margin=dict(l=0, r=0, t=40, b=0)
)
# Display map with Chrome
fig1.show()
pio.renderers.default = "chrome"

In [ ]:
# Carte 2 — Top hôtels par destination (une carte par ville)
with engine.connect() as conn:
    df_top5 = pd.read_sql(text("""
        SELECT city_id, city, lat, lon, temp_moy, rank
        FROM cities ORDER BY rank LIMIT 5
    """), conn)

    # 20 hôtels PAR ville 
    df_hotels_map = pd.read_sql(text("""
        SELECT h.hotel_name, h.score, h.price_eur, h.distance,
               h.lat, h.lon, c.city, c.temp_moy, c.city_id
        FROM hotels h
        JOIN cities c ON h.city_id = c.city_id
        WHERE h.city_id IN (
            SELECT city_id FROM cities ORDER BY rank LIMIT 5
        )
        AND h.score IS NOT NULL
        ORDER BY c.rank, h.score DESC
    """), conn)

# Garder les 20 meilleurs par ville
df_hotels_map = (
    df_hotels_map
    .groupby("city", group_keys=False)
    .apply(lambda x: x.nlargest(20, "score"))
    .reset_index(drop=True)
)

df_hotels_map["size_plot"] = df_hotels_map["score"] - df_hotels_map["score"].min() + 2

for _, city_row in df_top5.iterrows():
    city_name = city_row["city"]
    df_city   = df_hotels_map[df_hotels_map["city"] == city_name].copy()

    if df_city.empty:
        print(f"Aucun hôtel pour {city_name}")
        continue

    fig = go.Figure(go.Scattermapbox(
        lat=df_city["lat"].tolist(),
        lon=df_city["lon"].tolist(),
        mode="markers",
        marker=dict(
            size=(df_city["size_plot"] * 3).tolist(),
            color=df_city["score"].tolist(),
            colorscale="RdYlBu_r",
            cmin=df_hotels_map["score"].min(),
            cmax=df_hotels_map["score"].max(),
            showscale=True,
            colorbar=dict(title="Score Booking")
        ),
        text=df_city.apply(
            lambda r: (
                f"<b>{r['hotel_name']}</b><br>"
                f"Score : {r['score']}<br>"
                f"Prix : {r['price_eur']} €<br>"
                f"Distance : {r['distance']}"
            ), axis=1
        ).tolist(),
        hoverinfo="text"
    ))

    fig.update_layout(
        title=f"🏨 #{int(city_row['rank'])} {city_name} — {city_row['temp_moy']:.1f}°C · {len(df_city)} hôtels",
        mapbox=dict(
            style="carto-positron",
            center={"lat": df_city["lat"].mean(), "lon": df_city["lon"].mean()},
            zoom=13
        ),
        height=500,
        margin=dict(l=0, r=0, t=40, b=0)
    )


# Display map with Chrome
fig.show()
pio.renderers.default = "chrome"

In [ ]:
# Debug line
#df_hotels_map.head()
df_bormes = df_hotels_map[
    df_hotels_map["city"].str.contains("Bormes", case=False, na=False)
]

print(
    df_bormes[
        ["hotel_name", "score", "distance"]
    ].sort_values("score")
)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from sqlalchemy import text

# =====================================================
# Chargement des données
# =====================================================

with engine.connect() as conn:

    df_top5 = pd.read_sql(
        text("""
            SELECT city_id, city, lat, lon, temp_moy, rank
            FROM cities
            ORDER BY rank
            LIMIT 5
        """),
        conn
    )

    df_hotels_map = pd.read_sql(
        text("""
            SELECT
                h.hotel_name,
                h.score,
                h.price_eur,
                h.distance,
                h.lat,
                h.lon,
                c.city,
                c.temp_moy,
                c.city_id,
                c.rank
            FROM hotels h
            JOIN cities c
                ON h.city_id = c.city_id
            WHERE h.city_id IN (
                SELECT city_id
                FROM cities
                ORDER BY rank
                LIMIT 5
            )
            AND h.score IS NOT NULL
        """),
        conn
    )

# =====================================================
# Top 20 hôtels par ville
# =====================================================

df_hotels_map = (
    df_hotels_map
    .groupby("city", group_keys=False)
    .apply(lambda x: x.nlargest(20, "score"))
    .reset_index(drop=True)
)

# Taille des points
df_hotels_map["size_plot"] = (
    (df_hotels_map["score"] - df_hotels_map["score"].min()) * 4
    + 12
)

# =====================================================
# Carte interactive
# =====================================================

fig = go.Figure()

cities = (
    df_top5
    .sort_values("rank")
    ["city"]
    .tolist()
)

score_min = df_hotels_map["score"].min()
score_max = df_hotels_map["score"].max()

for i, city in enumerate(cities):

    df_city = df_hotels_map[
        df_hotels_map["city"] == city
    ]

    fig.add_trace(
        go.Scattermapbox(
            lat=df_city["lat"],
            lon=df_city["lon"],
            mode="markers",

            visible=(i == 0),

            name=city,

            marker=dict(
                size=df_city["size_plot"],
                color=df_city["score"],
                colorscale="Greens",
                cmin=score_min,
                cmax=score_max,
                showscale=(i == 0),
                colorbar=dict(
                    title="Score Booking"
                )
            ),

            text=df_city.apply(
                lambda r: (
                    f"<b>{r['hotel_name']}</b><br>"
                    f"⭐ Score : {r['score']}<br>"
                    f"💰 Prix : {r['price_eur']} €<br>"
                    f"📍 Distance : {r['distance']}"
                ),
                axis=1
            ),

            hoverinfo="text"
        )
    )

# =====================================================
# Menu déroulant
# =====================================================

buttons = []

for i, city in enumerate(cities):

    visible = [False] * len(cities)
    visible[i] = True

    city_temp = (
        df_top5.loc[
            df_top5["city"] == city,
            "temp_moy"
        ].iloc[0]
    )

    buttons.append(
        dict(
            label=city,
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": (
                        f"🏨 Top 20 hôtels — "
                        f"{city} ({city_temp:.1f}°C)"
                    )
                }
            ]
        )
    )

# =====================================================
# Centrage initial sur la première ville
# =====================================================

first_city = cities[0]

df_first = df_hotels_map[
    df_hotels_map["city"] == first_city
]

fig.update_layout(

    title=f"🏨 Top 20 hôtels — {first_city}",

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.01,
            y=1.08
        )
    ],

    mapbox=dict(
        style="carto-positron",
        center=dict(
            lat=df_first["lat"].mean(),
            lon=df_first["lon"].mean()
        ),
        zoom=12
    ),

    height=800,

    margin=dict(
        l=0,
        r=0,
        t=60,
        b=0
    )
)

fig.show()
pio.renderers.default = "chrome"